In [14]:
import pandas as pd
import numpy as np
import pathlib
import warnings

from SharedModules import input_dir, model_dir, output_dir
from xgboost import XGBClassifier
from sklearn.base import clone
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")

cv = KFold(n_splits=10, shuffle=True, random_state=42)

target = "Heart Disease"

X = pd.concat([pd.read_csv(f"{oof}/oof.csv", index_col=0) for oof in pathlib.Path(model_dir).iterdir() if oof.is_dir()], axis=1)
y = pd.read_csv(input_dir + "train.csv", index_col=0)[target].map({"Absence":0, "Presence": 1})
X_test = pd.concat([pd.read_csv(pred, index_col=0) for pred in pathlib.Path(output_dir).iterdir() if pred.is_file() and "meta_xgb" not in str(pred)], axis=1)

In [12]:
meta_xgb = XGBClassifier(
    random_state=42,
    n_estimators=800,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.9,
    enable_categorical=True,
    tree_method="hist",
    device="gpu",
    n_jobs=7,
)

In [15]:
oof = np.zeros(len(X))
oof_scores = []

for fold, (idx_tr, idx_val) in enumerate(cv.split(X, y)):
    X_tr, X_val = X.iloc[idx_tr], X.iloc[idx_val]
    y_tr, y_val = y.iloc[idx_tr], y.iloc[idx_val]

    model = clone(meta_xgb)
    model.fit(X_tr, y_tr)
    y_pred = model.predict_proba(X_val)[:, -1]
    score = roc_auc_score(y_val, y_pred)

    oof[idx_val] = y_pred
    oof_scores.append(score)

    print(f"fold {fold + 1} score: {score:.4f}")

print(f"Average Score: {np.mean(oof_scores): .4f}")

fold 1 score: 0.9558
fold 2 score: 0.9549
fold 3 score: 0.9555
fold 4 score: 0.9557
fold 5 score: 0.9548
fold 6 score: 0.9575
fold 7 score: 0.9552
fold 8 score: 0.9549
fold 9 score: 0.9556
fold 10 score: 0.9549
Average Score:  0.9555


In [16]:
meta_xgb.fit(X, y)
sub = pd.DataFrame(data=meta_xgb.predict_proba(X_test)[:, -1], index=X_test.index, columns=["meta_xgb"])
sub.to_csv(output_dir + "submission_meta_xgb.csv")

ValueError: feature_names mismatch: ['Keras', 'LGBM', 'StatsModels'] ['Keras', 'LGBM_Base', 'StatModels']
expected StatsModels, LGBM in input data
training data did not have the following fields: LGBM_Base, StatModels